In [4]:
import sys
sys.path.append("..") # Để import từ thư mục src (nếu cần)
import numpy as np
import cv2
import scipy.io
from pathlib import Path
from skimage.feature import hog
from tqdm.notebook import tqdm # Dùng để hiện thanh tiến trình (progress bar)

DATA_DIR = Path("../data/raw")
IMG_SIZE = (128, 128)

# Đọc lại nhãn và ID
labels = scipy.io.loadmat(DATA_DIR / "imagelabels.mat")["labels"][0]
setid = scipy.io.loadmat(DATA_DIR / "setid.mat")
train_ids = setid["trnid"][0] - 1
val_ids   = setid["valid"][0] - 1
test_ids  = setid["tstid"][0] - 1

In [5]:
def extract_features(img_id):
    """Đọc 1 ảnh và trả về vector đặc trưng (HOG + Color)"""
    path = DATA_DIR / "jpg" / f"image_{img_id+1:05d}.jpg"
    img = cv2.imread(str(path))
    img = cv2.resize(img, IMG_SIZE)
    
    # 1. Trích xuất HOG (dùng ảnh xám)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2)
    f_hog = hog(img_gray, orientations=9, pixels_per_cell=(8, 8),
                cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
    
    # 2. Trích xuất Color Histogram (dùng không gian màu HSV)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # Tính histogram cho kênh H (Hue) và S (Saturation), bỏ qua V (Value) để đỡ bị ảnh hưởng bởi ánh sáng
    hist_h = cv2.calcHist([img_hsv], [0], None, [16], [0, 180]).flatten()
    hist_s = cv2.calcHist([img_hsv], [1], None, [16], [0, 256]).flatten()
    f_color = np.concatenate([hist_h, hist_s])
    # Chuẩn hóa L1 để vector màu không bị quá lớn
    f_color = f_color / (f_color.sum() + 1e-7) 
    
    # Ghép nối 2 vector đặc trưng
    return np.concatenate([f_hog, f_color])

# Test thử với 1 ảnh để xem kích thước vector
test_feat = extract_features(train_ids[0])
print(f"Kích thước vector đặc trưng của 1 ảnh: {test_feat.shape}")

Kích thước vector đặc trưng của 1 ảnh: (8132,)


In [6]:
def build_dataset(ids, labels_array):
    X = []
    # Dùng tqdm để hiện thanh tiến trình
    for img_id in tqdm(ids, desc="Đang trích xuất đặc trưng"):
        feat = extract_features(img_id)
        X.append(feat)
    X = np.array(X)
    y = labels_array[ids] # Lấy nhãn tương ứng
    return X, y

print("1. Đang xử lý tập Train...")
X_train, y_train = build_dataset(train_ids, labels)

print("\n2. Đang xử lý tập Validation...")
X_val, y_val = build_dataset(val_ids, labels)

print("\n3. Đang xử lý tập Test...")
X_test, y_test = build_dataset(test_ids, labels)

print(f"\n✅ Hoàn tất! Kích thước dữ liệu:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

1. Đang xử lý tập Train...


Đang trích xuất đặc trưng:   0%|          | 0/1020 [00:00<?, ?it/s]


2. Đang xử lý tập Validation...


Đang trích xuất đặc trưng:   0%|          | 0/1020 [00:00<?, ?it/s]


3. Đang xử lý tập Test...


Đang trích xuất đặc trưng:   0%|          | 0/6149 [00:00<?, ?it/s]


✅ Hoàn tất! Kích thước dữ liệu:
X_train: (1020, 8132), y_train: (1020,)
X_val:   (1020, 8132), y_val:   (1020,)
X_test:  (6149, 8132), y_test:  (6149,)


In [7]:
# Tạo thư mục processed nếu chưa có
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Lưu lại
np.save(PROCESSED_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DIR / "y_train.npy", y_train)
np.save(PROCESSED_DIR / "X_val.npy", X_val)
np.save(PROCESSED_DIR / "y_val.npy", y_val)
np.save(PROCESSED_DIR / "X_test.npy", X_test)
np.save(PROCESSED_DIR / "y_test.npy", y_test)

print("💾 Đã lưu toàn bộ đặc trưng vào thư mục data/processed/")

💾 Đã lưu toàn bộ đặc trưng vào thư mục data/processed/
